In [1]:
"""
导出面板Logit所需数据
====================
从merged_data_scaled_CLEAN.parquet提取所需变量
输出: gme_logit.csv, amc_logit.csv
"""

import pandas as pd
import numpy as np
import json
import os

# ── 配置 ──────────────────────────────────────────────────────────────────────
GME_FILE    = 'data/merged_data_scaled_CLEAN.parquet'
AMC_FILE    = 'data/merged_data_amc_scaled_CLEAN.parquet'
GROUPS_FILE = 'data/feature_groups_CLEAN.json'
TARGET      = 'EWS_15min'

TRAIN_END  = '2021-01-25 23:59:59'
TEST_START = '2021-01-28 00:00:00'
TEST_END   = '2021-02-28 23:59:59'

LEAKAGE_FEATURES = [
    'is_episode_t', 'is_luld', 'is_anomaly',
    'is_price_spike', 'is_volume_surge',
    'thread_concentration', 'is_official_halt',
]

# ── 三个specification的特征组 ──────────────────────────────────────────────────
# Model 1: 只用市场微观结构特征
MARKET_FEATURES = [
    'returns', 'returns_std', 'returns_mean', 'returns_z',
    'volume', 'volume_std', 'volume_mean', 'volume_z',
    'trade_count', 'vwap', 'close', 'high', 'low', 'open',
    'returns_lag5min', 'volume_lag5min', 'returns_z_lag5min', 'volume_z_lag5min',
    'is_official_halt_lag1', 'is_official_halt_lag2',
]

# Model 2: 加入社交媒体量和情感（aggregate signals）
SOCIAL_AGG_FEATURES = [
    'post_volume', 'comment_volume', 'total_volume',
    'post_volume_15min', 'comment_volume_15min', 'total_volume_15min',
    'post_velocity', 'comment_velocity',
    'unique_users', 'user_growth',
    'sentiment_compound', 'sentiment_positive', 'sentiment_negative',
    'bullish_keywords', 'bearish_keywords', 'urgency_score',
    'is_burst', 'burst_intensity',
]

# Model 3: 加入网络拓扑/cascade/协调特征
NETWORK_COORD_FEATURES = [
    'node_count', 'edge_count', 'density',
    'avg_degree', 'max_degree',
    'avg_betweenness', 'max_betweenness',
    'avg_clustering', 'max_k_core',
    'avg_pagerank', 'max_pagerank',
    'cascade_size_mean', 'cascade_size_max',
    'cascade_depth_mean', 'cascade_breadth_mean',
    'structural_virality_mean', 'structural_virality_max',
    'adoption_speed_mean', 'cascade_count',
    'burstiness_coefficient', 'sync_posting_rate',
    'inter_arrival_mean', 'inter_arrival_std',
    'cross_thread_overlap', 'multi_thread_user_ratio',
    'coordinated_group_size',
    'exact_duplicate_rate', 'fuzzy_duplicate_rate',
    'avg_text_similarity',
]

ALL_FEATURES = MARKET_FEATURES + SOCIAL_AGG_FEATURES + NETWORK_COORD_FEATURES

def export_data(data_file, output_file, stock_name):
    print(f'\n处理 {stock_name}...')

    df = pd.read_parquet(data_file)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)

    # 加入时间变量
    df['date']      = df['timestamp'].dt.date.astype(str)
    df['hour']      = df['timestamp'].dt.hour
    df['minute']    = df['timestamp'].dt.minute
    df['year']      = df['timestamp'].dt.year
    df['month']     = df['timestamp'].dt.month
    df['dayofweek'] = df['timestamp'].dt.dayofweek

    # 只保留需要的列
    keep_cols = ['timestamp', 'date', 'hour', 'minute',
                 'year', 'month', 'dayofweek', TARGET]

    # 只保留实际存在的特征
    available = [f for f in ALL_FEATURES if f in df.columns]
    missing   = [f for f in ALL_FEATURES if f not in df.columns]
    if missing:
        print(f'  ⚠ 缺失特征（填0）: {len(missing)}个')
        for f in missing:
            df[f] = 0

    keep_cols = keep_cols + ALL_FEATURES
    df_export = df[keep_cols].copy()

    # 填充缺失值
    df_export[ALL_FEATURES] = df_export[ALL_FEATURES].fillna(0)

    # 加入split标记
    df_export['split'] = 'other'
    df_export.loc[df['timestamp'] <= TRAIN_END, 'split'] = 'train'
    df_export.loc[
        (df['timestamp'] >= TEST_START) & (df['timestamp'] <= TEST_END),
        'split'
    ] = 'test'

    df_export['stock'] = stock_name

    # 保存
    df_export.to_csv(output_file, index=False)

    print(f'  ✓ 保存: {output_file}')
    print(f'  维度: {df_export.shape}')
    print(f'  正样本: {int(df_export[TARGET].sum())} ({df_export[TARGET].mean()*100:.2f}%)')
    print(f'  训练集: {(df_export["split"]=="train").sum():,}')
    print(f'  测试集: {(df_export["split"]=="test").sum():,}')

    # 打印特征可用性
    print(f'\n  特征可用性:')
    print(f'    Market features:      {sum(f in df.columns for f in MARKET_FEATURES)}/{len(MARKET_FEATURES)}')
    print(f'    Social agg features:  {sum(f in df.columns for f in SOCIAL_AGG_FEATURES)}/{len(SOCIAL_AGG_FEATURES)}')
    print(f'    Network/coord:        {sum(f in df.columns for f in NETWORK_COORD_FEATURES)}/{len(NETWORK_COORD_FEATURES)}')

    return df_export

# ── 导出 ──────────────────────────────────────────────────────────────────────
os.makedirs('stata_data', exist_ok=True)

gme = export_data(GME_FILE, 'stata_data/gme_logit.csv', 'GME')
amc = export_data(AMC_FILE, 'stata_data/amc_logit.csv', 'AMC')

# 同时导出合并版本（备用）
combined = pd.concat([gme, amc], ignore_index=True)
combined.to_csv('stata_data/combined_logit.csv', index=False)
print(f'\n✓ 合并数据: stata_data/combined_logit.csv ({len(combined):,} rows)')

# 导出特征名列表供Stata使用
with open('stata_data/feature_lists.txt', 'w') as f:
    f.write('// Model 1: Market features only\n')
    f.write('global market_vars ' + ' '.join(MARKET_FEATURES) + '\n\n')
    f.write('// Model 2: + Social media aggregate\n')
    f.write('global social_agg_vars ' + ' '.join(SOCIAL_AGG_FEATURES) + '\n\n')
    f.write('// Model 3: + Network/coordination\n')
    f.write('global network_coord_vars ' + ' '.join(NETWORK_COORD_FEATURES) + '\n')

print('✓ 特征列表: stata_data/feature_lists.txt')
print('\n完成！下一步：在Stata里运行 panel_logit.do')


处理 GME...
  ✓ 保存: stata_data/gme_logit.csv
  维度: (59146, 77)
  正样本: 2565 (4.34%)
  训练集: 41,816
  测试集: 3,759

  特征可用性:
    Market features:      20/20
    Social agg features:  18/18
    Network/coord:        29/29

处理 AMC...
  ✓ 保存: stata_data/amc_logit.csv
  维度: (67510, 77)
  正样本: 2820 (4.18%)
  训练集: 47,299
  测试集: 3,922

  特征可用性:
    Market features:      20/20
    Social agg features:  18/18
    Network/coord:        29/29

✓ 合并数据: stata_data/combined_logit.csv (126,656 rows)
✓ 特征列表: stata_data/feature_lists.txt

完成！下一步：在Stata里运行 panel_logit.do
